In [1]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import numpy as np
import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)
tf.keras.utils.set_random_seed(42)

tf.config.experimental.enable_op_determinism()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [31]:
df = pd.read_csv("Churn_Modelling.csv")

print("\nDataset loaded successfully!")
print(df.head())


Dataset loaded successfully!
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93

In [32]:
# Standard dataset:
# Columns 3 to 12 = Input features
# Column 13 = Exited (target)

X = df.iloc[:, 3:13].copy()
y = df.iloc[:, 13].copy()

In [33]:
X = pd.get_dummies(
    X,
    columns=["Geography", "Gender"],
    drop_first=True,
    dtype=int
)

print("\nFeatures after encoding:")
print(X.head())


Features after encoding:
   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619   42       2       0.00              1          1   
1          608   41       1   83807.86              1          0   
2          502   42       8  159660.80              3          1   
3          699   39       1       0.00              2          0   
4          850   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  \
0               1        101348.88                  0                0   
1               1        112542.58                  0                1   
2               0        113931.57                  0                0   
3               0         93826.63                  0                0   
4               1         79084.10                  0                1   

   Gender_Male  
0            0  
1            0  
2            0  
3            0  
4            0  


In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
sc = StandardScaler()

X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [36]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(units=6, activation='relu'),
    Dense(units=6, activation='relu'),
    Dense(units=1, activation='sigmoid')
])

In [37]:
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("=== ANN MODEL PERFORMANCE ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step
=== ANN MODEL PERFORMANCE ===
Accuracy Score: 44.05%

Confusion Matrix:
 [[628 979]
 [140 253]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.39      0.53      1607
           1       0.21      0.64      0.31       393

    accuracy                           0.44      2000
   macro avg       0.51      0.52      0.42      2000
weighted avg       0.70      0.44      0.49      2000

